<a href="https://colab.research.google.com/github/AliMehdii/Memoire-2023/blob/master/Code/Version_01_Modeling_Colab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import os
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.callbacks import TensorBoard



In [4]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [5]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/mnist/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [7]:
!pip install wandb

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 KB 25.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 KB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 KB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 KB 20.3 MB/s eta 0:00:00
  Created wheel for pathtools: filename=pathtools-0.1.2-py3-none-any.whl size=8806 sha256=734c802244ab76bfdb3303e49a5b198a07d3d1268ce4166f3d80c6292fa4fe77
  Stored in directory: /root/.cache/pip/wheels/4c/8e/7e/72fbc243e1aeecae64a96875432e70d4e92f3d2d18123be004
Successfully built pathtools
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


In [8]:
import wandb
wandb.login()

ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 

··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [9]:
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint

In [10]:
# Start a run, tracking hyperparameters
wandb.init(
    # set the wandb project where this run will be logged
    project="Mermoire_2023_Version_01",

    # track hyperparameters and run metadata with wandb.config
    config={
        "dropout": 0.25,
        "dropout_2": 0.2,
        "activation": "softmax",
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 10,
        "batch_size": 32
    }
)

wandb: Currently logged in as: fleur. Use `wandb login --relogin` to force relogin


In [11]:
config = wandb.config

In [12]:
train_set = '/content/drive/My Drive/Cropped_Image_Sets/train'
val_set = '/content/drive/My Drive/Cropped_Image_Sets/val'
test_set = '/content/drive/My Drive/Cropped_Image_Sets/test'
model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"
IMAGE_SIZE = 128

In [27]:
def init_data(train_dir: str, valid_dir: str) -> tuple:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    
    return train_data, valid_data

In [28]:
train_data, valid_data = init_data(train_dir=train_set, valid_dir=val_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.


In [29]:
model_name = "My_model"

TensorBoard = TensorBoard(log_dir="logs\\{}".format(model_name))

In [30]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(rate=config.dropout),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(rate=config.dropout_2),
        tf.keras.layers.Dense(units=3, activation=config.activation)
    ])
    
    # Compile the model
    model.compile(
        loss=config.loss,
        optimizer=config.optimizer,
        metrics=[config.metric]
    )
    
    return model

In [31]:
rad_model = build_transfer_learning_model(
    base_model=ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
)

In [32]:
rad_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 2048)              23587712  
                                                                 
 flatten_2 (Flatten)         (None, 2048)              0         
                                                                 
 dropout_4 (Dropout)         (None, 2048)              0         
                                                                 
 batch_normalization_2 (Batc  (None, 2048)             8192      
 hNormalization)                                                 
                                                                 
 dense_4 (Dense)             (None, 128)               262272    
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                      

In [33]:
# Train the model for 10 epochs
rad_hist = rad_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=config.epoch,
    callbacks= [TensorBoard,
                WandbMetricsLogger(log_freq=5),
                WandbModelCheckpoint("models")]
)
wandb.finish()

Epoch 1/10
67/67 [==============================] - ETA: 0s - loss: 1.2495 - accuracy: 0.4571 

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 1202s 18s/step - loss: 1.2495 - accuracy: 0.4571 - val_loss: 1.0208 - val_accuracy: 0.4934
Epoch 2/10
67/67 [==============================] - ETA: 0s - loss: 0.8619 - accuracy: 0.6474

wandb: Adding directory to artifact (./models)... Done. 0.3s


67/67 [==============================] - 33s 497ms/step - loss: 0.8619 - accuracy: 0.6474 - val_loss: 0.9127 - val_accuracy: 0.6245
Epoch 3/10
67/67 [==============================] - ETA: 0s - loss: 0.5796 - accuracy: 0.7654

...

wandb: Adding directory to artifact (./models)... Done. 0.3s


67/67 [==============================] - 33s 503ms/step - loss: 0.2209 - accuracy: 0.9165 - val_loss: 0.5875 - val_accuracy: 0.7467
Epoch 10/10
67/67 [==============================] - ETA: 0s - loss: 0.1805 - accuracy: 0.9347

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 34s 509ms/step - loss: 0.1805 - accuracy: 0.9347 - val_loss: 0.7256 - val_accuracy: 0.6965


batch/accuracy,▁▁▁▂▂▃▄▄▅▆▆▆▆▆▆▆▆▇▇▆▆▇▇▇█▇▇▇▇▇▇▇███▇████
batch/batch_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,███▇▇▆▆▅▄▃▄▄▃▃▃▃▃▃▃▃▂▂▂▂▁▂▂▂▁▂▂▂▁▁▁▂▁▁▁▁
epoch/accuracy,▁▄▆▆▇▇▇▇██
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▃▂▂▂▁▁
epoch/val_accuracy,▁▅▃▄▇█▇▇█▇
epoch/val_loss,█▆▆▅▃▁▁▂▁▃
batch/accuracy,0.93371


In [ ]:
# IMAGE_SIZE = 256

# model_dir ="C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Models/RadImageNet-ResNet50_notop.h5"
# # base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, pooling="avg")
# base_model = ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
# base_model.summary()

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 3)  0           ['input_10[0][0]']               
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [ ]:
# base_model_config = base_model.get_config() 

In [ ]:
# channel_num = 1
# base_model_config["layers"][0]["config"]["batch_input_shape"] =(None, IMAGE_SIZE, IMAGE_SIZE, channel_num)


In [ ]:
# updated_base_model = Model.from_config(base_model_config)
# print(updated_base_model.summary())

Model: "resnet50"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 256, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 262, 262, 1)  0           ['input_9[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 128, 128, 64  3200        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                           

In [ ]:

# data = []
# Home = 'C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train'
# for folder in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train')):
#     for file in sorted(os.listdir('C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/train/'+folder)):
#         file_path = Home + '/' + folder + '/' + file

#         file_name = file.split('.')[0]
#         data.append([file_name, folder, file_path])

# train_set = pd.DataFrame(data, columns=['File_Name', 'Folder', 'File_Path'])
# print(train_set)
# train_set.to_csv('Train_set.csv')
